# Automated Marine Debris Detection using Deep Learning

## 1. Introduction and Problem Statement

## 2. Data Sources

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

### 3.2 Class Distribution

### 3.3 Image and Annotation Visualisation

### 3.4 Preprocessing and Data Splitting

In [2]:

import json
import random
from pathlib import Path
from collections import defaultdict, Counter


INPUT_JSON = Path("instance_version/instances_train_trashcan.json")

OUTPUT_DIR = Path("instance_version")

TEST_SIZE = 1000
MAX_ATTEMPTS = 200

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]
categories = data["categories"]

all_categories = {c["id"] for c in categories}

video_groups = defaultdict(list)

for img in images:
    filename = img["file_name"]
    video_id = filename.split("_frame")[0]
    video_groups[video_id].append(img)

image_categories = defaultdict(set)

for ann in annotations:
    image_categories[ann["image_id"]].add(ann["category_id"])

video_categories = {}

for video_id, group in video_groups.items():
    cats = set()
    for img in group:
        cats.update(image_categories[img["id"]])
    video_categories[video_id] = cats


def select_test_videos():
    video_ids = list(video_groups.keys())
    random.shuffle(video_ids)

    test = []
    id_list = set()

    for video in video_ids:
        group = video_groups[video]

        if len(test) < TEST_SIZE:
            test.extend(group)
            id_list.add(video)

    return id_list

selected_videos = None

for attempt in range(MAX_ATTEMPTS):
    candidate = select_test_videos()

    if candidate is None:
        continue

    test_cats = set()
    train_cats = set()

    for video, cats in video_categories.items():
        if video in candidate:
            test_cats.update(cats)
        else:
            train_cats.update(cats)

    if test_cats == all_categories and train_cats == all_categories:
        selected_videos = candidate
        break

if selected_videos is None:
    raise RuntimeError(
        "Failed to find a valid train/test split."
    )

train_images = []
test_images = []

for video, group in video_groups.items():
    if video in selected_videos:
        test_images.extend(group)
    else:
        train_images.extend(group)

train_ids = {img["id"] for img in train_images}
test_ids = {img["id"] for img in test_images}

train_annotations = [ann for ann in annotations if ann["image_id"] in train_ids]
test_annotations = [ann for ann in annotations if ann["image_id"] in test_ids]


def save_split(name, split_images, split_annotations):
    output = {
        "info": data.get("info", {}),
        "licenses": data.get("licenses", []),
        "images": split_images,
        "annotations": split_annotations,
        "categories": categories
    }

    path = OUTPUT_DIR / f"instances_{name}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f)

    print(f"Saved: {path}")


save_split("train", train_images, train_annotations)
save_split("test", test_images, test_annotations)


print("Train images:", len(train_images))
print("Test images:", len(test_images))
print("Train annotations:", len(train_annotations))
print("Test annotations:", len(test_annotations))


Saved: instance_version/instances_train.json
Saved: instance_version/instances_test.json
Train images: 4970
Test images: 1095
Train annotations: 7772
Test annotations: 1768


## 4. Models and Methods

### 4.1 Faster R-CNN

### 4.2 Mask R-CNN

In [ ]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor
)
from torchvision.models.detection.mask_rcnn import (
    MaskRCNNPredictor
)

def create_model(num_classes):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )
    return model


Baseline

In [ ]:
def load_baseline_model(device):
    model = maskrcnn_resnet50_fpn_v2(
            weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        )
    model.to(device=device)
    model.eval()

    return model

## 5. Experimental Setup

### 5.1 Training Configuration

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms.functional import pil_to_tensor
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F
from torchvision.ops.boxes import masks_to_boxes
from torchvision.io import read_image
from pycocotools.coco import COCO

class MarineDataset(Dataset):
    def __init__(self, image_dir, annotation_file, category_ids, transforms=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_file)
        self.image_ids = sorted(self.coco.getImgIds())
        self.category_ids = category_ids
        self.transforms = transforms
    
        self.cat_to_label = {
            cat_id: i + 1
            for i, cat_id in enumerate(category_ids)
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.coco.loadImgs(image_id)[0]

        image_path = os.path.join(self.image_dir, info["file_name"])
        image = read_image(image_path)

        ann_ids = self.coco.getAnnIds(imgIds=[image_id])
        annotations = self.coco.loadAnns(ann_ids)

        boxes, labels, masks = [], [], []

        for ann in annotations:

            mask = torch.as_tensor(self.coco.annToMask(ann), dtype=torch.uint8)

            box = masks_to_boxes(mask.unsqueeze(0))[0]

            boxes.append(box)
            labels.append(self.cat_to_label[ann["category_id"]])
            masks.append(mask)

        num_objs = len(labels)
        boxes = torch.stack(boxes).float()
        boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(image))
        labels = torch.as_tensor(labels, dtype=torch.int64)
        masks = torch.stack(masks)
        masks = tv_tensors.Mask(masks)
        image_id = int(image_id)
        area = ((boxes[:, 3] - boxes[:, 1]) *(boxes[:, 2] - boxes[:, 0]))
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        image = tv_tensors.Image(image)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


Transformer

In [ ]:
from torchvision.transforms import v2 as T
def get_transform(train):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

Dataset setup

In [ ]:

from help_funciton.engine import train_one_epoch, evaluate

# train on the accelerator or on the CPU, if an accelerator is not available
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')

# our dataset has two classes only - background and person
image_dir = "instance_version/train"
annotation_file = "instance_version/instances_train.json"
category_id = COCO(annotation_file).getCatIds()
num_classes = len(category_id) + 1
dataset = MarineDataset(image_dir, annotation_file, category_id, transforms=get_transform(train=False))
image_dir_val = "instance_version/val"
annotation_file_val = "instance_version/instances_val_trashcan.json"
category_id_val = COCO(annotation_file_val).getCatIds()
num_classes_val = len(category_id_val) + 1
dataset_val = MarineDataset(image_dir_val, annotation_file_val, category_id_val, transforms=get_transform(train=False))

dataset_trans = MarineDataset(image_dir, annotation_file, category_id, transforms=get_transform(train=True))


# define training and validation data loaders
def collate_fn(batch):
    return tuple(zip(*batch))
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

data_loader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

data_loader_trans = torch.utils.data.DataLoader(
    dataset_trans,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)



Train sgd+steplr

In [ ]:
model = create_model(num_classes)

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005,
    momentum=0.9
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_val, device=device)


Train adamw+cosineAnnealingLR

In [ ]:
model_adamw = create_model(num_classes)

model_adamw.to(device)

# construct an optimizer
params = [p for p in model_adamw.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    params,
     lr=0.0001,
    weight_decay=0.0001
)

num_epochs = 15

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

# let's train it just for 2 epochs


for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model_adamw, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model_adamw, data_loader_val, device=device)

train sgd+stepLR using trans data

In [ ]:

# get the model using our helper function
model_sgd_trans = create_model(num_classes)

# move model to the right device
model_sgd_trans.to(device)

# construct an optimizer
params = [p for p in model_sgd_trans.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005,
    momentum=0.9
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model_sgd_trans, optimizer, data_loader_trans, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model_sgd_trans, data_loader_val, device=device)


Test dataset setup

In [ ]:

image_dir_test = "instance_version/train"
annotation_file_test = "instance_version/instances_test.json"
category_id_test = COCO(annotation_file_test).getCatIds()
num_classes_test = len(category_id_test) + 1
dataset_test = MarineDataset(image_dir_test, annotation_file_test, category_id_test, transforms=get_transform(train=False))


data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
base_model = load_baseline_model(device=device)
base_model.to(device=device)
evaluater_base = evaluate(model=base_model, data_loader=data_loader_test, device=device)

In [ ]:
evaluater_sgd = evaluate(model=model, data_loader=data_loader_test, device=device)

In [ ]:
evaluater_adamw = evaluate(model=model_adamw, data_loader=data_loader_test, device=device)

In [ ]:
evaluater_sgd_trans = evaluate(model=model_sgd_trans, data_loader=data_loader_test, device=device)

### 5.2 Evaluation Metrics

## 6. Results

### 6.1 Quantitative Results

### 6.2 Qualitative Results

### 6.3 Model Comparison

## 7. Discussion

### 7.1 Error Analysis and Limitations

## 8. Conclusion and Future Work

## References